## Celda 0 — Configuración (Drive, rutas y Gradio)

In [5]:
%pip install -q gradio

In [6]:
from google.colab import drive
import os

# Montar Google Drive
drive.mount('/content/drive')

# Ruta base del proyecto en Drive
RUTA_BASE = '/content/drive/MyDrive/SGB/datos/'
os.makedirs(RUTA_BASE, exist_ok=True)

# Rutas de cada archivo de persistencia
RUTA_USUARIOS     = RUTA_BASE + 'usuarios.txt'
RUTA_LIBROS       = RUTA_BASE + 'libros.txt'
RUTA_CATEGORIAS   = RUTA_BASE + 'categorias.txt'
RUTA_PRESTAMOS    = RUTA_BASE + 'prestamos.txt'
RUTA_MULTAS       = RUTA_BASE + 'multas.txt'
RUTA_RESERVAS     = RUTA_BASE + 'reservas.txt'

Mounted at /content/drive


# SECCIÓN 1 — MODELO (dominio)
## Celda 1.1 — Enums y utilidades comunes

In [7]:
from enum import Enum
from datetime import datetime, date, timedelta

SEP = "|"                    # delimitador de campos en los archivos .txt
FORMATO_FECHA = "%Y-%m-%d"   # formato de fechas: AAAA-MM-DD


def _limpiar(texto):
    """Elimina del texto el delimitador y los saltos de línea para no corromper el archivo."""
    return str(texto).replace(SEP, "/").replace("\n", " ").strip()


def parse_fecha(texto):
    """Convierte un texto AAAA-MM-DD en un objeto date. Lanza ValueError si es inválido."""
    try:
        return datetime.strptime(str(texto).strip(), FORMATO_FECHA).date()
    except ValueError:
        raise ValueError(f"Fecha inválida '{texto}'. Use el formato AAAA-MM-DD (ej. 2026-10-26).")


def fmt_fecha(f):
    """Convierte un objeto date en texto AAAA-MM-DD."""
    return f.strftime(FORMATO_FECHA)


class TipoUsuarioEnum(Enum):
    """Tipos de usuario con sus reglas de préstamo (días de plazo y cupo máximo)."""
    ESTUDIANTE = "ESTUDIANTE"
    DOCENTE = "DOCENTE"
    EXTERNO = "EXTERNO"

    @property
    def dias_prestamo(self):
        """Días de plazo del préstamo según el tipo de usuario."""
        return {"ESTUDIANTE": 7, "DOCENTE": 15, "EXTERNO": 5}[self.value]

    @property
    def cupo(self):
        """Cantidad máxima de libros simultáneos según el tipo de usuario."""
        return {"ESTUDIANTE": 3, "DOCENTE": 5, "EXTERNO": 2}[self.value]


class EstadoPrestamoEnum(Enum):
    """Estados posibles de un préstamo."""
    ACTIVO = "ACTIVO"
    DEVUELTO = "DEVUELTO"
    VENCIDO = "VENCIDO"


class EstadoMultaEnum(Enum):
    """Estados posibles de una multa."""
    PENDIENTE = "PENDIENTE"
    PAGADA = "PAGADA"


class EstadoReservaEnum(Enum):
    """Estados posibles de una reserva."""
    ACTIVA = "ACTIVA"
    CUMPLIDA = "CUMPLIDA"
    CANCELADA = "CANCELADA"

## Celda 1.2 — Clase `Usuario`

In [8]:
class Usuario:
    """Persona registrada en la biblioteca (estudiante, docente o externo)."""
    ACTIVO = "ACTIVO"
    SUSPENDIDO = "SUSPENDIDO"

    def __init__(self, num_documento, nombre, tipo, correo, telefono, estado="ACTIVO"):
        """Crea un usuario. `tipo` es un TipoUsuarioEnum; `estado` es ACTIVO o SUSPENDIDO."""
        self.num_documento = _limpiar(num_documento)
        self.nombre = _limpiar(nombre)
        self.tipo = tipo
        self.correo = _limpiar(correo)
        self.telefono = _limpiar(telefono)
        self.estado = estado

    def esta_activo(self):
        """Indica si el usuario está habilitado para prestar."""
        return self.estado == Usuario.ACTIVO

    def to_linea(self):
        """Serializa el usuario como una línea de texto delimitada."""
        return SEP.join([self.num_documento, self.nombre, self.tipo.value,
                         self.correo, self.telefono, self.estado])

    @staticmethod
    def from_linea(linea):
        """Reconstruye un Usuario a partir de una línea de texto."""
        p = linea.split(SEP)
        return Usuario(p[0], p[1], TipoUsuarioEnum(p[2]), p[3], p[4], p[5])

## Celda 1.3 — Clase `Categoria`

In [9]:
class Categoria:
    """Categoría temática que agrupa libros."""

    def __init__(self, codigo, nombre, descripcion=""):
        """Crea una categoría con código único, nombre y descripción."""
        self.codigo = _limpiar(codigo).upper()
        self.nombre = _limpiar(nombre)
        self.descripcion = _limpiar(descripcion)

    def to_linea(self):
        """Serializa la categoría como una línea de texto delimitada."""
        return SEP.join([self.codigo, self.nombre, self.descripcion])

    @staticmethod
    def from_linea(linea):
        """Reconstruye una Categoria a partir de una línea de texto."""
        p = linea.split(SEP)
        return Categoria(p[0], p[1], p[2])

## Celda 1.4 — Clase `Libro`

In [10]:
class Libro:
    """Libro del catálogo con control de ejemplares totales y disponibles."""

    def __init__(self, isbn, titulo, autor, cod_categoria, anio, total_ejemplares, disponibles=None):
        """Crea un libro. Si no se indica `disponibles`, se asume igual al total."""
        self.isbn = _limpiar(isbn)
        self.titulo = _limpiar(titulo)
        self.autor = _limpiar(autor)
        self.cod_categoria = _limpiar(cod_categoria).upper()
        self.anio = int(anio)
        self.total_ejemplares = int(total_ejemplares)
        self.disponibles = self.total_ejemplares if disponibles is None else int(disponibles)

    def to_linea(self):
        """Serializa el libro como una línea de texto delimitada."""
        return SEP.join([self.isbn, self.titulo, self.autor, self.cod_categoria,
                         str(self.anio), str(self.total_ejemplares), str(self.disponibles)])

    @staticmethod
    def from_linea(linea):
        """Reconstruye un Libro a partir de una línea de texto."""
        p = linea.split(SEP)
        return Libro(p[0], p[1], p[2], p[3], int(p[4]), int(p[5]), int(p[6]))

## Celda 1.5 — Clase `Prestamo`

In [11]:
class Prestamo:
    """Préstamo de un libro a un usuario."""

    def __init__(self, codigo, isbn, num_documento, fecha_prestamo, fecha_limite,
                 fecha_devolucion="", estado=EstadoPrestamoEnum.ACTIVO):
        """Crea un préstamo. `fecha_devolucion` queda vacía mientras no se devuelva."""
        self.codigo = codigo
        self.isbn = isbn
        self.num_documento = num_documento
        self.fecha_prestamo = fecha_prestamo
        self.fecha_limite = fecha_limite
        self.fecha_devolucion = fecha_devolucion
        self.estado = estado

    def esta_pendiente(self):
        """True si el libro aún no ha sido devuelto."""
        return self.fecha_devolucion == ""

    def to_linea(self):
        """Serializa el préstamo como una línea de texto delimitada."""
        return SEP.join([self.codigo, self.isbn, self.num_documento, self.fecha_prestamo,
                         self.fecha_limite, self.fecha_devolucion, self.estado.value])

    @staticmethod
    def from_linea(linea):
        """Reconstruye un Prestamo a partir de una línea de texto."""
        p = linea.split(SEP)
        return Prestamo(p[0], p[1], p[2], p[3], p[4], p[5], EstadoPrestamoEnum(p[6]))

## Celda 1.6 — Clase `Multa`

In [12]:
class Multa:
    """Multa generada por devolución tardía de un préstamo."""

    def __init__(self, codigo, cod_prestamo, dias_retraso, valor, estado=EstadoMultaEnum.PENDIENTE):
        """Crea una multa asociada a un préstamo."""
        self.codigo = codigo
        self.cod_prestamo = cod_prestamo
        self.dias_retraso = int(dias_retraso)
        self.valor = int(valor)
        self.estado = estado

    def to_linea(self):
        """Serializa la multa como una línea de texto delimitada."""
        return SEP.join([self.codigo, self.cod_prestamo, str(self.dias_retraso),
                         str(self.valor), self.estado.value])

    @staticmethod
    def from_linea(linea):
        """Reconstruye una Multa a partir de una línea de texto."""
        p = linea.split(SEP)
        return Multa(p[0], p[1], int(p[2]), int(p[3]), EstadoMultaEnum(p[4]))

## Celda 1.7 — Clase `Reserva`

In [13]:
class Reserva:
    """Reserva de un libro sin ejemplares disponibles (se atiende por orden de llegada)."""

    def __init__(self, codigo, isbn, num_documento, fecha_reserva, estado=EstadoReservaEnum.ACTIVA):
        """Crea una reserva de un usuario sobre un libro."""
        self.codigo = codigo
        self.isbn = isbn
        self.num_documento = num_documento
        self.fecha_reserva = fecha_reserva
        self.estado = estado

    def to_linea(self):
        """Serializa la reserva como una línea de texto delimitada."""
        return SEP.join([self.codigo, self.isbn, self.num_documento,
                         self.fecha_reserva, self.estado.value])

    @staticmethod
    def from_linea(linea):
        """Reconstruye una Reserva a partir de una línea de texto."""
        p = linea.split(SEP)
        return Reserva(p[0], p[1], p[2], p[3], EstadoReservaEnum(p[4]))

# SECCIÓN 2 — REPOSITORIOS (persistencia en archivos de texto)
## Celda 2.1 — `ArchivoUtil` y repositorio base

In [14]:
import os
import uuid


class ArchivoUtil:
    """Utilidades de bajo nivel para leer y escribir archivos de texto."""

    @staticmethod
    def leer_lineas(ruta):
        """Devuelve la lista de líneas no vacías del archivo (lista vacía si no existe)."""
        if not os.path.exists(ruta):
            return []
        with open(ruta, "r", encoding="utf-8") as f:
            return [l.strip("\n") for l in f if l.strip()]

    @staticmethod
    def escribir_linea(ruta, linea):
        """Agrega una línea al final del archivo."""
        with open(ruta, "a", encoding="utf-8") as f:
            f.write(linea + "\n")

    @staticmethod
    def reescribir(ruta, lineas):
        """Reemplaza todo el contenido del archivo por las líneas dadas."""
        with open(ruta, "w", encoding="utf-8") as f:
            for l in lineas:
                f.write(l + "\n")

    @staticmethod
    def generar_codigo(prefijo):
        """Genera un código único con el prefijo dado (ej. PRE-3FA9C1)."""
        return f"{prefijo}-{uuid.uuid4().hex[:6].upper()}"


class RepositorioBase:
    """Comportamiento común de los repositorios: leer todo, guardar y reemplazar por clave."""

    def __init__(self, ruta, clase):
        """Recibe la ruta del archivo y la clase del modelo que persiste."""
        self.ruta = ruta
        self.clase = clase

    def find_all(self):
        """Devuelve todos los registros del archivo como objetos del modelo."""
        return [self.clase.from_linea(l) for l in ArchivoUtil.leer_lineas(self.ruta)]

    def guardar(self, obj):
        """Agrega un nuevo registro al archivo."""
        ArchivoUtil.escribir_linea(self.ruta, obj.to_linea())

    def _reemplazar(self, obj, atributo_clave):
        """Reemplaza el registro cuya clave coincide con la del objeto dado."""
        items = self.find_all()
        for i, x in enumerate(items):
            if getattr(x, atributo_clave) == getattr(obj, atributo_clave):
                items[i] = obj
                break
        ArchivoUtil.reescribir(self.ruta, [x.to_linea() for x in items])

## Celda 2.2 — `UsuarioRepository`

In [15]:
class UsuarioRepository(RepositorioBase):
    """Persistencia de usuarios en usuarios.txt."""

    def __init__(self, ruta=None):
        """Usa por defecto RUTA_USUARIOS."""
        super().__init__(ruta or RUTA_USUARIOS, Usuario)

    def find_by_documento(self, doc):
        """Busca un usuario por número de documento; None si no existe."""
        return next((u for u in self.find_all() if u.num_documento == doc.strip()), None)

    def actualizar(self, u):
        """Actualiza los datos de un usuario existente."""
        self._reemplazar(u, "num_documento")

## Celda 2.3 — `CategoriaRepository`

In [16]:
class CategoriaRepository(RepositorioBase):
    """Persistencia de categorías en categorias.txt."""

    def __init__(self, ruta=None):
        """Usa por defecto RUTA_CATEGORIAS."""
        super().__init__(ruta or RUTA_CATEGORIAS, Categoria)

    def find_by_codigo(self, cod):
        """Busca una categoría por código; None si no existe."""
        return next((c for c in self.find_all() if c.codigo == cod.strip().upper()), None)

## Celda 2.4 — `LibroRepository`

In [17]:
class LibroRepository(RepositorioBase):
    """Persistencia de libros en libros.txt."""

    def __init__(self, ruta=None):
        """Usa por defecto RUTA_LIBROS."""
        super().__init__(ruta or RUTA_LIBROS, Libro)

    def find_by_isbn(self, isbn):
        """Busca un libro por ISBN; None si no existe."""
        return next((l for l in self.find_all() if l.isbn == isbn.strip()), None)

    def find_by_titulo(self, txt):
        """Libros cuyo título contiene el texto (sin distinguir mayúsculas)."""
        t = txt.strip().lower()
        return [l for l in self.find_all() if t in l.titulo.lower()]

    def find_by_categoria(self, cod):
        """Libros de una categoría."""
        return [l for l in self.find_all() if l.cod_categoria == cod.strip().upper()]

    def actualizar(self, l):
        """Actualiza los datos de un libro existente."""
        self._reemplazar(l, "isbn")

## Celda 2.5 — `PrestamoRepository`

In [18]:
class PrestamoRepository(RepositorioBase):
    """Persistencia de préstamos en prestamos.txt."""

    def __init__(self, ruta=None):
        """Usa por defecto RUTA_PRESTAMOS."""
        super().__init__(ruta or RUTA_PRESTAMOS, Prestamo)

    def find_by_codigo(self, cod):
        """Busca un préstamo por código; None si no existe."""
        return next((p for p in self.find_all() if p.codigo == cod.strip().upper()), None)

    def find_by_usuario(self, doc):
        """Préstamos de un usuario."""
        return [p for p in self.find_all() if p.num_documento == doc.strip()]

    def find_by_libro(self, isbn):
        """Préstamos de un libro."""
        return [p for p in self.find_all() if p.isbn == isbn.strip()]

    def actualizar(self, p):
        """Actualiza un préstamo existente."""
        self._reemplazar(p, "codigo")

## Celda 2.6 — `MultaRepository`

In [19]:
class MultaRepository(RepositorioBase):
    """Persistencia de multas en multas.txt."""

    def __init__(self, prestamo_repo, ruta=None):
        """Recibe el PrestamoRepository para poder asociar multas con usuarios."""
        super().__init__(ruta or RUTA_MULTAS, Multa)
        self.prestamo_repo = prestamo_repo

    def find_by_codigo(self, cod):
        """Busca una multa por código; None si no existe."""
        return next((m for m in self.find_all() if m.codigo == cod.strip().upper()), None)

    def find_by_prestamo(self, cod):
        """Busca la multa generada por un préstamo; None si no existe."""
        return next((m for m in self.find_all() if m.cod_prestamo == cod), None)

    def find_pendientes_usuario(self, doc):
        """Multas PENDIENTES de un usuario (a través de sus préstamos)."""
        codigos = {p.codigo for p in self.prestamo_repo.find_by_usuario(doc)}
        return [m for m in self.find_all()
                if m.cod_prestamo in codigos and m.estado == EstadoMultaEnum.PENDIENTE]

    def actualizar_estado(self, cod, estado):
        """Cambia el estado de una multa (recibe el texto del estado)."""
        m = self.find_by_codigo(cod)
        if m:
            m.estado = EstadoMultaEnum(estado)
            self._reemplazar(m, "codigo")

## Celda 2.7 — `ReservaRepository`

In [20]:
class ReservaRepository(RepositorioBase):
    """Persistencia de reservas en reservas.txt."""

    def __init__(self, ruta=None):
        """Usa por defecto RUTA_RESERVAS."""
        super().__init__(ruta or RUTA_RESERVAS, Reserva)

    def find_by_codigo(self, cod):
        """Busca una reserva por código; None si no existe."""
        return next((r for r in self.find_all() if r.codigo == cod.strip().upper()), None)

    def find_by_libro(self, isbn):
        """Reservas de un libro en orden de llegada (orden de registro en el archivo)."""
        return [r for r in self.find_all() if r.isbn == isbn.strip()]

    def find_by_usuario(self, doc):
        """Reservas de un usuario."""
        return [r for r in self.find_all() if r.num_documento == doc.strip()]

    def actualizar_estado(self, cod, estado):
        """Cambia el estado de una reserva (recibe el texto del estado)."""
        r = self.find_by_codigo(cod)
        if r:
            r.estado = EstadoReservaEnum(estado)
            self._reemplazar(r, "codigo")

# SECCIÓN 3 — SERVICIOS (lógica de negocio)
## Celda 3.1 — `UsuarioService`

In [21]:
class UsuarioService:
    """Reglas de negocio para la gestión de usuarios."""

    def __init__(self, usuario_repo):
        """Recibe el repositorio de usuarios."""
        self.repo = usuario_repo

    def registrar(self, num_documento, nombre, tipo, correo, telefono):
        """Registra un usuario nuevo validando campos obligatorios y documento único."""
        if not num_documento.strip() or not nombre.strip():
            raise ValueError("El documento y el nombre son obligatorios.")
        if "@" not in correo:
            raise ValueError("El correo no es válido.")
        if self.repo.find_by_documento(num_documento):
            raise ValueError(f"Ya existe un usuario con documento {num_documento}.")
        u = Usuario(num_documento, nombre, TipoUsuarioEnum(tipo), correo, telefono)
        self.repo.guardar(u)
        return u

    def buscar_por_documento(self, doc):
        """Devuelve el usuario con ese documento o lanza ValueError si no existe."""
        u = self.repo.find_by_documento(doc)
        if not u:
            raise ValueError(f"No existe un usuario con documento {doc}.")
        return u

    def actualizar(self, doc, correo=None, telefono=None, estado=None):
        """Actualiza correo, teléfono y/o estado (los valores vacíos no se modifican)."""
        u = self.buscar_por_documento(doc)
        if correo and correo.strip():
            if "@" not in correo:
                raise ValueError("El correo no es válido.")
            u.correo = correo.strip()
        if telefono and telefono.strip():
            u.telefono = telefono.strip()
        if estado:
            u.estado = estado
        self.repo.actualizar(u)
        return u

    def listar_todos(self):
        """Lista todos los usuarios registrados."""
        return self.repo.find_all()

## Celda 3.2 — `LibroService`

In [22]:
class LibroService:
    """Reglas de negocio para el catálogo de libros y categorías."""

    def __init__(self, libro_repo, categoria_repo):
        """Recibe los repositorios de libros y categorías."""
        self.libro_repo = libro_repo
        self.categoria_repo = categoria_repo

    def registrar_categoria(self, codigo, nombre, descripcion=""):
        """Registra una categoría nueva con código único."""
        if not codigo.strip() or not nombre.strip():
            raise ValueError("El código y el nombre de la categoría son obligatorios.")
        if self.categoria_repo.find_by_codigo(codigo):
            raise ValueError(f"Ya existe la categoría {codigo.upper()}.")
        c = Categoria(codigo, nombre, descripcion)
        self.categoria_repo.guardar(c)
        return c

    def listar_categorias(self):
        """Lista todas las categorías."""
        return self.categoria_repo.find_all()

    def registrar(self, isbn, titulo, autor, cod_categoria, anio, ejemplares):
        """Registra un libro validando ISBN único, categoría existente y cantidades."""
        if not isbn.strip() or not titulo.strip() or not autor.strip():
            raise ValueError("ISBN, título y autor son obligatorios.")
        if not cod_categoria or not self.categoria_repo.find_by_codigo(cod_categoria):
            raise ValueError("Seleccione una categoría existente (regístrela primero si no está).")
        if self.libro_repo.find_by_isbn(isbn):
            raise ValueError(f"Ya existe un libro con ISBN {isbn}.")
        anio, ejemplares = int(anio), int(ejemplares)
        if not (1000 <= anio <= date.today().year):
            raise ValueError("El año de publicación no es válido.")
        if ejemplares < 1:
            raise ValueError("El número de ejemplares debe ser al menos 1.")
        l = Libro(isbn, titulo, autor, cod_categoria, anio, ejemplares)
        self.libro_repo.guardar(l)
        return l

    def buscar_por_isbn(self, isbn):
        """Busca un libro por ISBN (lista con 0 o 1 elementos)."""
        l = self.libro_repo.find_by_isbn(isbn)
        return [l] if l else []

    def buscar_por_titulo(self, txt):
        """Busca libros por coincidencia parcial de título."""
        return self.libro_repo.find_by_titulo(txt)

    def listar_por_categoria(self, cod):
        """Lista los libros de una categoría."""
        return self.libro_repo.find_by_categoria(cod)

    def listar_disponibles(self):
        """Lista los libros con al menos un ejemplar disponible."""
        return [l for l in self.libro_repo.find_all() if l.disponibles > 0]

    def listar_todos(self):
        """Lista todo el catálogo."""
        return self.libro_repo.find_all()

## Celda 3.3 — `MultaService` *(se define antes que PrestamoService, que lo usa)*

In [23]:
class MultaService:
    """Cálculo, pago y consulta de multas."""
    VALOR_DIA = 1000   # COP por día de retraso

    def __init__(self, multa_repo):
        """Recibe el repositorio de multas."""
        self.repo = multa_repo

    def calcular_multa(self, prestamo):
        """Si el préstamo se devolvió tarde, crea y guarda la multa; si no, devuelve None."""
        if prestamo.esta_pendiente():
            return None
        dias = (parse_fecha(prestamo.fecha_devolucion) - parse_fecha(prestamo.fecha_limite)).days
        if dias <= 0 or self.repo.find_by_prestamo(prestamo.codigo):
            return None
        m = Multa(ArchivoUtil.generar_codigo("MUL"), prestamo.codigo, dias, dias * self.VALOR_DIA)
        self.repo.guardar(m)
        return m

    def pagar(self, cod_multa):
        """Registra el pago de una multa pendiente."""
        m = self.repo.find_by_codigo(cod_multa)
        if not m:
            raise ValueError(f"No existe la multa {cod_multa}.")
        if m.estado == EstadoMultaEnum.PAGADA:
            raise ValueError("Esa multa ya fue pagada.")
        self.repo.actualizar_estado(m.codigo, EstadoMultaEnum.PAGADA.value)
        m.estado = EstadoMultaEnum.PAGADA
        return m

    def listar_pendientes_usuario(self, doc):
        """Multas pendientes de un usuario."""
        return self.repo.find_pendientes_usuario(doc)

## Celda 3.4 — `PrestamoService`

In [24]:
class PrestamoService:
    """Reglas de negocio de préstamos y devoluciones."""

    def __init__(self, prestamo_repo, libro_repo, usuario_repo, reserva_repo, multa_service):
        """Recibe los repositorios necesarios y el servicio de multas."""
        self.repo = prestamo_repo
        self.libro_repo = libro_repo
        self.usuario_repo = usuario_repo
        self.reserva_repo = reserva_repo
        self.multa_service = multa_service

    def prestar(self, num_documento, isbn, fecha_prestamo):
        """Registra un préstamo validando usuario activo, multas, cupo y disponibilidad (con cola de reservas)."""
        f = parse_fecha(fecha_prestamo)
        u = self.usuario_repo.find_by_documento(num_documento)
        if not u:
            raise ValueError("El usuario no existe.")
        if not u.esta_activo():
            raise ValueError("El usuario está suspendido.")
        if self.multa_service.listar_pendientes_usuario(u.num_documento):
            raise ValueError("El usuario tiene multas pendientes.")
        pendientes = [p for p in self.repo.find_by_usuario(u.num_documento) if p.esta_pendiente()]
        if len(pendientes) >= u.tipo.cupo:
            raise ValueError(f"El usuario alcanzó su cupo ({u.tipo.cupo} libros para {u.tipo.value.lower()}).")
        if any(p.isbn == isbn.strip() for p in pendientes):
            raise ValueError("El usuario ya tiene prestado este libro.")
        libro = self.libro_repo.find_by_isbn(isbn)
        if not libro:
            raise ValueError("El libro no existe.")
        if libro.disponibles <= 0:
            raise ValueError("No hay ejemplares disponibles. Puede crear una reserva.")

        # Orden de llegada: los ejemplares libres se asignan primero a quienes reservaron antes
        cola = [r for r in self.reserva_repo.find_by_libro(libro.isbn) if r.estado == EstadoReservaEnum.ACTIVA]
        reserva_usuario = next((r for r in cola if r.num_documento == u.num_documento), None)
        if reserva_usuario:
            if cola.index(reserva_usuario) >= libro.disponibles:
                raise ValueError("Aún no es su turno en la cola de reservas de este libro.")
        elif len(cola) >= libro.disponibles:
            raise ValueError("Los ejemplares libres están reservados por otros usuarios. Puede reservar.")

        limite = f + timedelta(days=u.tipo.dias_prestamo)
        p = Prestamo(ArchivoUtil.generar_codigo("PRE"), libro.isbn, u.num_documento,
                     fmt_fecha(f), fmt_fecha(limite))
        self.repo.guardar(p)
        libro.disponibles -= 1
        self.libro_repo.actualizar(libro)
        if reserva_usuario:
            self.reserva_repo.actualizar_estado(reserva_usuario.codigo, EstadoReservaEnum.CUMPLIDA.value)
        return p

    def devolver(self, cod_prestamo, fecha_devolucion):
        """Registra la devolución, repone el ejemplar y genera la multa si hubo retraso. Devuelve (prestamo, multa|None)."""
        p = self.buscar_por_codigo(cod_prestamo)
        if not p.esta_pendiente():
            raise ValueError("Ese préstamo ya fue devuelto.")
        f = parse_fecha(fecha_devolucion)
        if f < parse_fecha(p.fecha_prestamo):
            raise ValueError("La fecha de devolución no puede ser anterior a la del préstamo.")
        p.fecha_devolucion = fmt_fecha(f)
        p.estado = EstadoPrestamoEnum.DEVUELTO
        self.repo.actualizar(p)
        libro = self.libro_repo.find_by_isbn(p.isbn)
        if libro:
            libro.disponibles = min(libro.disponibles + 1, libro.total_ejemplares)
            self.libro_repo.actualizar(libro)
        multa = self.multa_service.calcular_multa(p)
        return p, multa

    def buscar_por_codigo(self, cod):
        """Devuelve el préstamo con ese código o lanza ValueError."""
        p = self.repo.find_by_codigo(cod)
        if not p:
            raise ValueError(f"No existe el préstamo {cod}.")
        return p

    def historial_usuario(self, doc, solo_pendientes=False):
        """Préstamos de un usuario en orden cronológico."""
        lista = sorted(self.repo.find_by_usuario(doc), key=lambda p: p.fecha_prestamo)
        return [p for p in lista if p.esta_pendiente()] if solo_pendientes else lista

    def listar_vencidos(self, fecha):
        """Préstamos no devueltos cuya fecha límite es anterior a la fecha dada (y los marca VENCIDO)."""
        f = parse_fecha(fecha)
        vencidos = []
        for p in self.repo.find_all():
            if p.esta_pendiente() and parse_fecha(p.fecha_limite) < f:
                if p.estado != EstadoPrestamoEnum.VENCIDO:
                    p.estado = EstadoPrestamoEnum.VENCIDO
                    self.repo.actualizar(p)
                vencidos.append(p)
        return vencidos

## Celda 3.5 — `ReservaService`

In [25]:
class ReservaService:
    """Reglas de negocio de reservas (cola por orden de llegada)."""

    def __init__(self, reserva_repo, libro_repo, usuario_repo):
        """Recibe los repositorios de reservas, libros y usuarios."""
        self.repo = reserva_repo
        self.libro_repo = libro_repo
        self.usuario_repo = usuario_repo

    def reservar(self, num_documento, isbn, fecha):
        """Crea una reserva si el libro no tiene ejemplares libres sin comprometer."""
        f = parse_fecha(fecha)
        u = self.usuario_repo.find_by_documento(num_documento)
        if not u:
            raise ValueError("El usuario no existe.")
        if not u.esta_activo():
            raise ValueError("El usuario está suspendido.")
        libro = self.libro_repo.find_by_isbn(isbn)
        if not libro:
            raise ValueError("El libro no existe.")
        activas = self.listar_por_libro(libro.isbn)
        if any(r.num_documento == u.num_documento for r in activas):
            raise ValueError("El usuario ya tiene una reserva activa para este libro.")
        if libro.disponibles - len(activas) > 0:
            raise ValueError("Hay ejemplares disponibles: realice el préstamo directamente.")
        r = Reserva(ArchivoUtil.generar_codigo("RES"), libro.isbn, u.num_documento, fmt_fecha(f))
        self.repo.guardar(r)
        return r

    def cancelar(self, cod_reserva):
        """Cancela una reserva activa."""
        r = self.repo.find_by_codigo(cod_reserva)
        if not r:
            raise ValueError(f"No existe la reserva {cod_reserva}.")
        if r.estado != EstadoReservaEnum.ACTIVA:
            raise ValueError("Solo se pueden cancelar reservas activas.")
        self.repo.actualizar_estado(r.codigo, EstadoReservaEnum.CANCELADA.value)
        r.estado = EstadoReservaEnum.CANCELADA
        return r

    def listar_por_libro(self, isbn):
        """Reservas activas de un libro, en orden de llegada."""
        return [r for r in self.repo.find_by_libro(isbn) if r.estado == EstadoReservaEnum.ACTIVA]

    def listar_por_usuario(self, doc):
        """Todas las reservas de un usuario."""
        return self.repo.find_by_usuario(doc)

## Celda 3.6 — `AnalisisDatosService`

In [26]:
from collections import Counter


class AnalisisDatosService:
    """Indicadores estadísticos de uso de la colección."""

    def __init__(self, prestamo_repo, libro_repo, categoria_repo, usuario_repo, multa_repo):
        """Recibe los repositorios que alimentan los análisis."""
        self.prestamo_repo = prestamo_repo
        self.libro_repo = libro_repo
        self.categoria_repo = categoria_repo
        self.usuario_repo = usuario_repo
        self.multa_repo = multa_repo

    def libros_mas_prestados(self, desde, hasta, top=10):
        """Lista de (isbn, título, cantidad) de los libros más prestados entre dos fechas."""
        d, h = parse_fecha(desde), parse_fecha(hasta)
        cuenta = Counter(p.isbn for p in self.prestamo_repo.find_all()
                         if d <= parse_fecha(p.fecha_prestamo) <= h)
        resultado = []
        for isbn, n in cuenta.most_common(int(top)):
            libro = self.libro_repo.find_by_isbn(isbn)
            resultado.append((isbn, libro.titulo if libro else "(eliminado)", n))
        return resultado

    def prestamos_por_categoria(self):
        """Lista de (categoría, cantidad, porcentaje) de préstamos por categoría."""
        libros = {l.isbn: l.cod_categoria for l in self.libro_repo.find_all()}
        nombres = {c.codigo: c.nombre for c in self.categoria_repo.find_all()}
        cuenta = Counter(libros.get(p.isbn, "?") for p in self.prestamo_repo.find_all())
        total = sum(cuenta.values())
        return [(nombres.get(cod, cod), n, round(100 * n / total, 1))
                for cod, n in cuenta.most_common()] if total else []

    def morosidad_por_tipo(self, fecha_corte=None):
        """Lista de (tipo, préstamos, morosos, tasa%) — moroso: devuelto tarde o vencido sin devolver."""
        corte = parse_fecha(fecha_corte) if fecha_corte else date.today()
        tipos = {u.num_documento: u.tipo for u in self.usuario_repo.find_all()}
        total, morosos = Counter(), Counter()
        for p in self.prestamo_repo.find_all():
            tipo = tipos.get(p.num_documento)
            if tipo is None:
                continue
            total[tipo] += 1
            limite = parse_fecha(p.fecha_limite)
            tarde = (parse_fecha(p.fecha_devolucion) > limite) if not p.esta_pendiente() else (corte > limite)
            if tarde:
                morosos[tipo] += 1
        return [(t.value, total[t], morosos[t], round(100 * morosos[t] / total[t], 1) if total[t] else 0.0)
                for t in TipoUsuarioEnum]

    def duracion_promedio_prestamo(self):
        """Duración promedio real (días) de los préstamos devueltos; None si no hay datos."""
        dias = [(parse_fecha(p.fecha_devolucion) - parse_fecha(p.fecha_prestamo)).days
                for p in self.prestamo_repo.find_all() if not p.esta_pendiente()]
        return round(sum(dias) / len(dias), 2) if dias else None

    def reporte_multas(self):
        """Diccionario con cantidad de multas y valores generado, pagado y pendiente (COP)."""
        multas = self.multa_repo.find_all()
        pagado = sum(m.valor for m in multas if m.estado == EstadoMultaEnum.PAGADA)
        pendiente = sum(m.valor for m in multas if m.estado == EstadoMultaEnum.PENDIENTE)
        return {"cantidad": len(multas), "generado": pagado + pendiente,
                "pagado": pagado, "pendiente": pendiente}

## Celda 3.7 — Ensamblaje de dependencias (repositorios + servicios)

In [27]:
# Repositorios
usuario_repo   = UsuarioRepository()
categoria_repo = CategoriaRepository()
libro_repo     = LibroRepository()
prestamo_repo  = PrestamoRepository()
multa_repo     = MultaRepository(prestamo_repo)
reserva_repo   = ReservaRepository()

# Servicios
usuario_service  = UsuarioService(usuario_repo)
libro_service    = LibroService(libro_repo, categoria_repo)
multa_service    = MultaService(multa_repo)
prestamo_service = PrestamoService(prestamo_repo, libro_repo, usuario_repo, reserva_repo, multa_service)
reserva_service  = ReservaService(reserva_repo, libro_repo, usuario_repo)
analisis_service = AnalisisDatosService(prestamo_repo, libro_repo, categoria_repo, usuario_repo, multa_repo)
print("✅ Capas de modelo, repositorios y servicios listas.")

✅ Capas de modelo, repositorios y servicios listas.


# SECCIÓN 4 — INTERFAZ DE USUARIO (Gradio)
En esta versión los menús de `print()`/`input()` se reemplazan por una interfaz web con pestañas (una por módulo).
## Celda 4.0 — Funciones auxiliares de la interfaz

In [28]:
import functools
import gradio as gr

COLS_USUARIO  = ["Documento", "Nombre", "Tipo", "Correo", "Teléfono", "Estado"]
COLS_CATEG    = ["Código", "Nombre", "Descripción"]
COLS_LIBRO    = ["ISBN", "Título", "Autor", "Categoría", "Año", "Total", "Disponibles"]
COLS_PRESTAMO = ["Código", "ISBN", "Documento", "F. préstamo", "F. límite", "F. devolución", "Estado"]
COLS_MULTA    = ["Código", "Préstamo", "Días retraso", "Valor (COP)", "Estado"]
COLS_RESERVA  = ["Código", "ISBN", "Documento", "F. reserva", "Estado"]


def hoy():
    """Fecha actual en texto AAAA-MM-DD."""
    return fmt_fecha(date.today())


def cop(valor):
    """Formatea un valor como pesos colombianos (COP 1.000)."""
    return "COP " + f"{valor:,}".replace(",", ".")


def filas_usuarios(lista):
    """Convierte usuarios en filas de tabla."""
    return [[u.num_documento, u.nombre, u.tipo.value, u.correo, u.telefono, u.estado] for u in lista]


def filas_categorias(lista):
    """Convierte categorías en filas de tabla."""
    return [[c.codigo, c.nombre, c.descripcion] for c in lista]


def filas_libros(lista):
    """Convierte libros en filas de tabla (con el nombre de la categoría)."""
    nombres = {c.codigo: c.nombre for c in categoria_repo.find_all()}
    return [[l.isbn, l.titulo, l.autor, nombres.get(l.cod_categoria, l.cod_categoria),
             l.anio, l.total_ejemplares, l.disponibles] for l in lista]


def filas_prestamos(lista):
    """Convierte préstamos en filas de tabla."""
    return [[p.codigo, p.isbn, p.num_documento, p.fecha_prestamo, p.fecha_limite,
             p.fecha_devolucion or "—", p.estado.value] for p in lista]


def filas_multas(lista):
    """Convierte multas en filas de tabla."""
    return [[m.codigo, m.cod_prestamo, m.dias_retraso, cop(m.valor), m.estado.value] for m in lista]


def filas_reservas(lista):
    """Convierte reservas en filas de tabla."""
    return [[r.codigo, r.isbn, r.num_documento, r.fecha_reserva, r.estado.value] for r in lista]


def opciones_categorias():
    """Opciones (etiqueta, valor) para los desplegables de categorías."""
    return [(f"{c.nombre} ({c.codigo})", c.codigo) for c in categoria_repo.find_all()]


def seguro(n_extra):
    """Decorador: captura ValueError y devuelve el mensaje ❌ más `n_extra` salidas sin cambios."""
    def deco(fn):
        @functools.wraps(fn)
        def envoltura(*args):
            try:
                return fn(*args)
            except (ValueError, TypeError) as e:
                return (f"❌ {e}",) + tuple(gr.update() for _ in range(n_extra))
        return envoltura
    return deco

## Celda 4.1 — Módulo Usuarios

In [31]:
@seguro(1)
def ui_registrar_usuario(doc, nombre, tipo, correo, tel):
    """Registra un usuario y refresca la tabla."""
    u = usuario_service.registrar(doc, nombre, tipo, correo, tel)
    return f" Usuario **{u.nombre}** registrado ({u.tipo.value}).", filas_usuarios(usuario_service.listar_todos())


@seguro(1)
def ui_consultar_usuario(doc):
    """Consulta un usuario por documento."""
    return " Usuario encontrado.", filas_usuarios([usuario_service.buscar_por_documento(doc)])


@seguro(1)
def ui_actualizar_usuario(doc, correo, tel, estado):
    """Actualiza contacto y estado de un usuario."""
    u = usuario_service.actualizar(doc, correo, tel, estado)
    return f" Datos de **{u.nombre}** actualizados.", filas_usuarios([u])


def ui_listar_usuarios():
    """Lista todos los usuarios."""
    return filas_usuarios(usuario_service.listar_todos())


def tab_usuarios():
    """Construye la pestaña de usuarios."""
    with gr.Tab(" Usuarios"):
        gr.Markdown("### Registrar usuario")
        with gr.Row():
            doc = gr.Textbox(label="Documento")
            nombre = gr.Textbox(label="Nombre completo")
            tipo = gr.Dropdown([t.value for t in TipoUsuarioEnum], value="ESTUDIANTE", label="Tipo")
        with gr.Row():
            correo = gr.Textbox(label="Correo")
            tel = gr.Textbox(label="Teléfono")
        btn_reg = gr.Button("Registrar", variant="primary")
        gr.Markdown("### Consultar / actualizar por documento")
        with gr.Row():
            c_doc = gr.Textbox(label="Documento")
            c_correo = gr.Textbox(label="Nuevo correo (opcional)")
            c_tel = gr.Textbox(label="Nuevo teléfono (opcional)")
            c_estado = gr.Dropdown(["ACTIVO", "SUSPENDIDO"], label="Nuevo estado (opcional)")
        with gr.Row():
            btn_con = gr.Button("Consultar")
            btn_act = gr.Button("Actualizar")
            btn_lis = gr.Button("Listar todos")
        msg = gr.Markdown()
        tabla = gr.Dataframe(headers=COLS_USUARIO, interactive=False, wrap=True)
        btn_reg.click(ui_registrar_usuario, [doc, nombre, tipo, correo, tel], [msg, tabla])
        btn_con.click(ui_consultar_usuario, [c_doc], [msg, tabla])
        btn_act.click(ui_actualizar_usuario, [c_doc, c_correo, c_tel, c_estado], [msg, tabla])
        btn_lis.click(ui_listar_usuarios, None, [tabla])

## Celda 4.2 — Módulo Libros y categorías

In [30]:
@seguro(3)
def ui_registrar_categoria(cod, nombre, desc):
    """Registra una categoría y actualiza los desplegables."""
    c = libro_service.registrar_categoria(cod, nombre, desc)
    op = opciones_categorias()
    return (f" Categoría **{c.nombre}** registrada.", filas_categorias(libro_service.listar_categorias()),
            gr.update(choices=op), gr.update(choices=op))


@seguro(1)
def ui_registrar_libro(isbn, titulo, autor, cat, anio, ejemplares):
    """Registra un libro en el catálogo."""
    l = libro_service.registrar(isbn, titulo, autor, cat, anio, ejemplares)
    return f" Libro **{l.titulo}** registrado con {l.total_ejemplares} ejemplar(es).", filas_libros(libro_service.listar_todos())


@seguro(1)
def ui_buscar_isbn(isbn):
    """Busca libros por ISBN."""
    r = libro_service.buscar_por_isbn(isbn)
    return f"🔎 {len(r)} resultado(s).", filas_libros(r)


@seguro(1)
def ui_buscar_titulo(txt):
    """Busca libros por título."""
    r = libro_service.buscar_por_titulo(txt)
    return f"🔎 {len(r)} resultado(s).", filas_libros(r)


@seguro(1)
def ui_buscar_categoria(cod):
    """Busca libros por categoría."""
    r = libro_service.listar_por_categoria(cod or "")
    return f"🔎 {len(r)} resultado(s).", filas_libros(r)


def ui_libros_disponibles():
    """Lista libros con ejemplares disponibles."""
    r = libro_service.listar_disponibles()
    return f"📚 {len(r)} libro(s) con ejemplares disponibles.", filas_libros(r)


def ui_refrescar_categorias():
    """Recarga las categorías en ambos desplegables y la tabla."""
    op = opciones_categorias()
    return filas_categorias(libro_service.listar_categorias()), gr.update(choices=op), gr.update(choices=op)


def tab_libros():
    """Construye la pestaña de libros y categorías."""
    with gr.Tab("📚 Libros y categorías"):
        gr.Markdown("### Categorías")
        with gr.Row():
            k_cod = gr.Textbox(label="Código (ej. LIT)")
            k_nom = gr.Textbox(label="Nombre")
            k_desc = gr.Textbox(label="Descripción")
        btn_cat = gr.Button("Registrar categoría", variant="primary")
        tabla_cat = gr.Dataframe(headers=COLS_CATEG, interactive=False, value=filas_categorias(libro_service.listar_categorias()))
        gr.Markdown("### Registrar libro")
        with gr.Row():
            isbn = gr.Textbox(label="ISBN")
            titulo = gr.Textbox(label="Título")
            autor = gr.Textbox(label="Autor")
        with gr.Row():
            cat = gr.Dropdown(opciones_categorias(), label="Categoría")
            anio = gr.Number(label="Año de publicación", precision=0, value=2020)
            ejem = gr.Number(label="Ejemplares", precision=0, value=1)
        btn_lib = gr.Button("Registrar libro", variant="primary")
        gr.Markdown("### Consultas")
        with gr.Row():
            b_isbn = gr.Textbox(label="Por ISBN")
            b_tit = gr.Textbox(label="Por título (parcial)")
            b_cat = gr.Dropdown(opciones_categorias(), label="Por categoría")
        with gr.Row():
            btn_bi = gr.Button("Buscar ISBN")
            btn_bt = gr.Button("Buscar título")
            btn_bc = gr.Button("Buscar categoría")
            btn_disp = gr.Button("Listar disponibles")
            btn_ref = gr.Button(" Refrescar categorías")
        msg = gr.Markdown()
        tabla = gr.Dataframe(headers=COLS_LIBRO, interactive=False, wrap=True)
        btn_cat.click(ui_registrar_categoria, [k_cod, k_nom, k_desc], [msg, tabla_cat, cat, b_cat])
        btn_lib.click(ui_registrar_libro, [isbn, titulo, autor, cat, anio, ejem], [msg, tabla])
        btn_bi.click(ui_buscar_isbn, [b_isbn], [msg, tabla])
        btn_bt.click(ui_buscar_titulo, [b_tit], [msg, tabla])
        btn_bc.click(ui_buscar_categoria, [b_cat], [msg, tabla])
        btn_disp.click(ui_libros_disponibles, None, [msg, tabla])
        btn_ref.click(ui_refrescar_categorias, None, [tabla_cat, cat, b_cat])

## Celda 4.3 — Módulo Préstamos

In [ ]:
@seguro(1)
def ui_prestar(doc, isbn, fecha):
    """Registra un préstamo."""
    p = prestamo_service.prestar(doc, isbn, fecha)
    return f"✅ Préstamo **{p.codigo}** registrado. Fecha límite: **{p.fecha_limite}**.", filas_prestamos([p])


@seguro(1)
def ui_devolver(cod, fecha):
    """Registra una devolución e informa si generó multa."""
    p, m = prestamo_service.devolver(cod, fecha)
    txt = f"✅ Devolución del préstamo **{p.codigo}** registrada."
    if m:
        txt += f"\n\n⚠️ Devolución tardía: **{m.dias_retraso} día(s)** → multa **{m.codigo}** por **{cop(m.valor)}**."
    return txt, filas_prestamos([p])


@seguro(1)
def ui_consultar_prestamo(cod):
    """Consulta un préstamo por código."""
    return "✅ Préstamo encontrado.", filas_prestamos([prestamo_service.buscar_por_codigo(cod)])


@seguro(1)
def ui_historial(doc, solo_pend):
    """Historial cronológico de préstamos de un usuario."""
    usuario_service.buscar_por_documento(doc)
    r = prestamo_service.historial_usuario(doc, solo_pend)
    return f"📖 {len(r)} préstamo(s).", filas_prestamos(r)


@seguro(1)
def ui_vencidos(fecha):
    """Lista los préstamos vencidos a una fecha."""
    r = prestamo_service.listar_vencidos(fecha)
    return f"⏰ {len(r)} préstamo(s) vencido(s) al {fecha}.", filas_prestamos(r)


def tab_prestamos():
    """Construye la pestaña de préstamos."""
    with gr.Tab("🔁 Préstamos"):
        gr.Markdown("### Registrar préstamo")
        with gr.Row():
            doc = gr.Textbox(label="Documento del usuario")
            isbn = gr.Textbox(label="ISBN del libro")
            fecha = gr.Textbox(label="Fecha de préstamo", value=hoy())
        btn_pre = gr.Button("Prestar", variant="primary")
        gr.Markdown("### Registrar devolución")
        with gr.Row():
            cod = gr.Textbox(label="Código del préstamo (PRE-XXXXXX)")
            f_dev = gr.Textbox(label="Fecha de devolución", value=hoy())
        with gr.Row():
            btn_dev = gr.Button("Devolver", variant="primary")
            btn_cod = gr.Button("Consultar préstamo")
        gr.Markdown("### Consultas")
        with gr.Row():
            h_doc = gr.Textbox(label="Documento (historial)")
            h_pend = gr.Checkbox(label="Solo no devueltos")
            f_venc = gr.Textbox(label="Vencidos a la fecha", value=hoy())
        with gr.Row():
            btn_his = gr.Button("Historial del usuario")
            btn_ven = gr.Button("Listar vencidos")
        msg = gr.Markdown()
        tabla = gr.Dataframe(headers=COLS_PRESTAMO, interactive=False, wrap=True)
        btn_pre.click(ui_prestar, [doc, isbn, fecha], [msg, tabla])
        btn_dev.click(ui_devolver, [cod, f_dev], [msg, tabla])
        btn_cod.click(ui_consultar_prestamo, [cod], [msg, tabla])
        btn_his.click(ui_historial, [h_doc, h_pend], [msg, tabla])
        btn_ven.click(ui_vencidos, [f_venc], [msg, tabla])

## Celda 4.4 — Módulo Reservas y multas

In [35]:
@seguro(1)
def ui_reservar(doc, isbn, fecha):
    """Crea una reserva."""
    r = reserva_service.reservar(doc, isbn, fecha)
    pos = len(reserva_service.listar_por_libro(isbn))
    return f" Reserva **{r.codigo}** creada. Posición en la cola: **{pos}**.", filas_reservas(reserva_service.listar_por_libro(isbn))


@seguro(1)
def ui_cancelar_reserva(cod):
    """Cancela una reserva."""
    r = reserva_service.cancelar(cod)
    return f" Reserva **{r.codigo}** cancelada.", filas_reservas([r])


@seguro(1)
def ui_reservas_libro(isbn):
    """Lista la cola de reservas activas de un libro."""
    r = reserva_service.listar_por_libro(isbn)
    return f"📋 {len(r)} reserva(s) activa(s), en orden de llegada.", filas_reservas(r)


@seguro(1)
def ui_multas_pendientes(doc):
    """Lista las multas pendientes de un usuario."""
    usuario_service.buscar_por_documento(doc)
    m = multa_service.listar_pendientes_usuario(doc)
    return f"💰 {len(m)} multa(s) pendiente(s) por **{cop(sum(x.valor for x in m))}**.", filas_multas(m)


@seguro(1)
def ui_pagar_multa(cod):
    """Registra el pago de una multa."""
    m = multa_service.pagar(cod)
    return f" Multa **{m.codigo}** pagada ({cop(m.valor)}).", filas_multas([m])


def tab_reservas_multas():
    """Construye la pestaña de reservas y multas."""
    with gr.Tab(" Reservas y multas"):
        gr.Markdown("### Reservas")
        with gr.Row():
            doc = gr.Textbox(label="Documento")
            isbn = gr.Textbox(label="ISBN")
            fecha = gr.Textbox(label="Fecha de reserva", value=hoy())
        with gr.Row():
            btn_res = gr.Button("Reservar", variant="primary")
            btn_lis = gr.Button("Ver cola del ISBN")
        with gr.Row():
            c_cod = gr.Textbox(label="Código de reserva (RES-XXXXXX)")
            btn_can = gr.Button("Cancelar reserva")
        msg_r = gr.Markdown()
        tabla_r = gr.Dataframe(headers=COLS_RESERVA, interactive=False)
        gr.Markdown("### Multas (COP 1.000 por día de retraso, calculadas automáticamente al devolver)")
        with gr.Row():
            m_doc = gr.Textbox(label="Documento del usuario")
            btn_mp = gr.Button("Ver multas pendientes")
        with gr.Row():
            m_cod = gr.Textbox(label="Código de multa (MUL-XXXXXX)")
            btn_pag = gr.Button("Registrar pago", variant="primary")
        msg_m = gr.Markdown()
        tabla_m = gr.Dataframe(headers=COLS_MULTA, interactive=False)
        btn_res.click(ui_reservar, [doc, isbn, fecha], [msg_r, tabla_r])
        btn_lis.click(ui_reservas_libro, [isbn], [msg_r, tabla_r])
        btn_can.click(ui_cancelar_reserva, [c_cod], [msg_r, tabla_r])
        btn_mp.click(ui_multas_pendientes, [m_doc], [msg_m, tabla_m])
        btn_pag.click(ui_pagar_multa, [m_cod], [msg_m, tabla_m])

## Celda 4.5 — Módulo Análisis de datos

In [34]:
@seguro(1)
def ui_mas_prestados(desde, hasta, top):
    """Libros más prestados en un periodo."""
    r = analisis_service.libros_mas_prestados(desde, hasta, top)
    return f" Periodo {desde} a {hasta}.", [[i + 1, a, b, c] for i, (a, b, c) in enumerate(r)]


def ui_por_categoria():
    """Distribución de préstamos por categoría."""
    return [list(f) for f in analisis_service.prestamos_por_categoria()]


@seguro(1)
def ui_morosidad(fecha):
    """Tasa de morosidad por tipo de usuario."""
    return " Moroso = devuelto tarde o vencido sin devolver.", [list(f) for f in analisis_service.morosidad_por_tipo(fecha)]


def ui_duracion():
    """Duración promedio de los préstamos devueltos."""
    d = analisis_service.duracion_promedio_prestamo()
    return "No hay préstamos devueltos todavía." if d is None else f"⏱️ Duración promedio real: **{d} días**."


def ui_reporte_multas():
    """Reporte de multas generadas, pagadas y pendientes."""
    r = analisis_service.reporte_multas()
    return (f"**Multas registradas:** {r['cantidad']}\n\n- Generado: **{cop(r['generado'])}**\n"
            f"- Pagado: **{cop(r['pagado'])}**\n- Pendiente: **{cop(r['pendiente'])}**")


def tab_analisis():
    """Construye la pestaña de análisis de datos."""
    with gr.Tab(" Análisis"):
        gr.Markdown("### Libros más prestados")
        with gr.Row():
            desde = gr.Textbox(label="Desde", value=fmt_fecha(date.today() - timedelta(days=365)))
            hasta = gr.Textbox(label="Hasta", value=hoy())
            top = gr.Number(label="Top", value=10, precision=0)
        btn_mp = gr.Button("Calcular")
        msg_mp = gr.Markdown()
        t_mp = gr.Dataframe(headers=["#", "ISBN", "Título", "Préstamos"], interactive=False)
        gr.Markdown("### Préstamos por categoría")
        btn_cat = gr.Button("Calcular")
        t_cat = gr.Dataframe(headers=["Categoría", "Préstamos", "%"], interactive=False)
        gr.Markdown("### Morosidad por tipo de usuario")
        f_corte = gr.Textbox(label="Fecha de corte", value=hoy())
        btn_mor = gr.Button("Calcular")
        msg_mor = gr.Markdown()
        t_mor = gr.Dataframe(headers=["Tipo", "Préstamos", "Morosos", "Tasa %"], interactive=False)
        gr.Markdown("### Duración promedio y reporte de multas")
        with gr.Row():
            btn_dur = gr.Button("Duración promedio")
            btn_rep = gr.Button("Reporte de multas")
        out = gr.Markdown()
        btn_mp.click(ui_mas_prestados, [desde, hasta, top], [msg_mp, t_mp])
        btn_cat.click(ui_por_categoria, None, [t_cat])
        btn_mor.click(ui_morosidad, [f_corte], [msg_mor, t_mor])
        btn_dur.click(ui_duracion, None, [out])
        btn_rep.click(ui_reporte_multas, None, [out])

## Celda 4.6 — Interfaz principal (equivale a `menu_principal()`)

In [38]:
def menu_principal():
    """Construye la interfaz principal con una pestaña por módulo y la devuelve (gr.Blocks)."""
    with gr.Blocks(title="SGB — Sistema de Gestión Bibliotecaria") as demo:
        gr.Markdown("#  Sistema de Gestión Bibliotecaria (SGB) — v1")
        tab_usuarios()
        tab_libros()
        tab_prestamos()
        tab_reservas_multas()
        tab_analisis()
    return demo

# SECCIÓN 5 — EJECUCIÓN PRINCIPAL
## Celda 5.1 — Iniciar el sistema

In [39]:
demo = menu_principal()
demo.launch(debug=True)

NameError: name 'tab_prestamos' is not defined